<div>
<center><img src="../assets/Flux-logo.svg" width="400"/>
</div>

# Chapter 2: Python Submission API 🐍️

Flux also provides first-class python bindings which can be used to submit jobs programmatically.  For this chapter in the tutorial, we will be using Python and you can interact with cells in the notebook. To get started with the Python tutorial, change directory to:

```bash
cd /home/ubuntu/tutorial/ch2
```

### Importing the flux package

Flux requires Python to build, so if you have Flux installed, you have at least one Python installation that works. However, you can also `pip install flux-python` to get the `flux` package in a side-installation of Python. The Python SDK interacts with the current Flux instance via the Flux handle. Let's show how to test that.

In [3]:
import flux 
print(flux.Flux())

In [4]:
import os
import json
import flux
import flux.job
from flux.job import JobspecV1
from flux.job.JobID import JobID

### `flux.job.JobspecV1`

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> The JobspecV1 class provides an easy means to create job specifications
</div>

Flux represents work as a standard called the [Jobspec](https://flux-framework.readthedocs.io/projects/flux-rfc/en/latest/spec_25.html). While you could write YAML or JSON, it's much easier to use provided Python functions that take high level metadata (command, resources, etc) to generate them. We can then replicate our previous example of submitting multiple heterogeneous jobs using these Python helpers, and testing that Flux co-schedules them.

In [33]:
# connect to the running Flux instance
f = flux.Flux()

# Create the Jobspec from a command to run a python script, and specify resources
jobspec = JobspecV1.from_command(
    command=["sleep", "0"], num_tasks=1, num_nodes=1, cores_per_task=1
)

# Attributes like environment, cwd, attributes input/output, queue and ruration can be set directly.
jobspec.environment = {'TMPDIR': '/tmp/'}
jobid = flux.job.submit(f, jobspec)

# When we submit, we get back the job identifier (JobID)
print(f'{jobid.f58 = }')

jobid.f58 = 'ƒkLsZEaNK'


We get a job ID, and the command will _block_ until a job ID is assigned. But, perhaps we want more information about the job, too.

In [34]:
# Maybe we want a bit more information about this job
info = flux.job.result(f, jobid)
print(json.dumps(info.to_dict(), indent=4))

{
    "t_run": 1754855108.0022883,
    "t_cleanup": 1754855108.0174737,
    "duration": 0.0,
    "result": "COMPLETED",
    "waitstatus": 0,
    "id": 95699806978048,
    "t_submit": 1754855107.9783614,
    "runtime": 0.015185356140136719,
    "returncode": 0,
    "dependencies": [],
    "annotations": {},
    "exception": {
        "occurred": false
    }
}


Once we create the job, when we submit it in Python we get back a job identifier or jobid. We can then interact with the Flux handle, a connection to Flux, to get information about that job.

### `flux.job.get_job(handle, jobid)`

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> The `get_job` function makes it easy to get job information like state, status, and timings
</div>


In [36]:
# Let's get a flux.job.JobID Python object for our job.
fid = JobID(jobid.f58)

# Why? We can represent it in many ways!
print(f"🎉️ Hooray, we just submitted {fid}!")
print(f"🎉️ Hooray, we just submitted {fid.emoji}!")
print(f"🎉️ Hooray, we just submitted {fid.words}!")
print(f"🎉️ Hooray, we just submitted {fid.real}!")
print(f"🎉️ Hooray, we just submitted {fid.dec}!")
print(f"🎉️ Hooray, we just submitted {fid.denominator}!")
print(f"🎉️ Hooray, we just submitted {fid.hex}!")
print(f"🎉️ Hooray, we just submitted {fid.dothex}!")
print(f"🎉️ Hooray, we just submitted {fid.f58}!")
print(f"🎉️ Hooray, we just submitted {fid.f58plain}!\n")

# Here is how to get your info. The first argument is the flux handle, then the jobid
jobinfo = flux.job.get_job(f, fid)
print(jobinfo)

🎉️ Hooray, we just submitted ƒkLsZEaNK!
🎉️ Hooray, we just submitted 😄🎪🍜💚🚙📓!
🎉️ Hooray, we just submitted teacher-alien-grace--lima-alarm-academy!
🎉️ Hooray, we just submitted 95699806978048!
🎉️ Hooray, we just submitted 95699806978048!
🎉️ Hooray, we just submitted 1!
🎉️ Hooray, we just submitted 0x5709d9000000!
🎉️ Hooray, we just submitted 0000.5709.d900.0000!
🎉️ Hooray, we just submitted ƒkLsZEaNK!
🎉️ Hooray, we just submitted fkLsZEaNK!

{'t_depend': 1754855107.990325, 't_run': 1754855108.0022883, 't_cleanup': 1754855108.0174737, 't_inactive': 1754855108.018468, 'duration': 0.0, 'expiration': 9223372036.0, 'name': 'sleep', 'cwd': '', 'queue': '', 'project': '', 'bank': '', 'ntasks': 1, 'ncores': 1, 'nnodes': 1, 'priority': 16, 'ranks': '3', 'nodelist': 'ip-10-0-20-251', 'success': True, 'result': 'COMPLETED', 'waitstatus': 0, 'id': JobID(95699806978048), 't_submit': 1754855107.9783614, 't_remaining': 0.0, 'state': 'INACTIVE', 'username': 'ubuntu', 'userid': 1000, 'urgency': 16, 'run

Look at what came from `.get_job()`. It didn't block, but the job isn't complete yet -- that `t_run` is 0, which makes no sense, because the run time should be around 5 seconds. Notice the status is also 'SCHED', meaning that the job is being scheduled, 

You can now run `flux jobs` to see the jobs that we submit from Python.

In [37]:
!flux jobs -a --name="sleep"

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
   ƒkLsZEaNK ubuntu   sleep      CD      1      1   0.015s ip-10-0-20-251
   ƒZMn6D4FZ ubuntu   sleep      CD      1      1   0.465s ip-10-0-20-251
   ƒZMn4j4y8 ubuntu   sleep      CD      1      1   0.466s ip-10-0-20-251
   ƒZMn3F5h4 ubuntu   sleep      CD      1      1   0.467s ip-10-0-20-251
   ƒZMn4j4yE ubuntu   sleep      CD      1      1   0.466s ip-10-0-20-251
   ƒZMn4j4yQ ubuntu   sleep      CD      1      1   0.466s ip-10-0-20-251
   ƒZMn3F5h8 ubuntu   sleep      CD      1      1   0.467s ip-10-0-20-251
   ƒZMn1m6R4 ubuntu   sleep      CD      1      1   0.467s ip-10-0-20-251
   ƒZMn4j4yL ubuntu   sleep      CD      1      1   0.466s ip-10-0-20-251
   ƒZMn6D4FR ubuntu   sleep      CD      1      1   0.466s ip-10-0-20-251
   ƒZMn6D4Fc ubuntu   sleep      CD      1      1   0.465s ip-10-0-20-251
   ƒZMn6D4Fh ubuntu   sleep      CD      1      1   0.461s ip-10-0-20-251
   ƒZMn3F5gr ubuntu   sleep      CD      1      

Under the hood, the `Jobspec` class is creating a YAML document that ultimately gets serialized as JSON and sent to Flux for ingestion, validation, queueing, scheduling, and eventually execution.  We can dump the raw JSON jobspec that is submitted, where we can see the exact resources requested and the task set to be executed on those resources.

In [38]:
print(jobspec.dumps(indent=4))

{
    "resources": [
        {
            "type": "node",
            "count": 1,
            "with": [
                {
                    "type": "slot",
                    "count": 1,
                    "with": [
                        {
                            "type": "core",
                            "count": 1
                        }
                    ],
                    "label": "task"
                }
            ]
        }
    ],
    "tasks": [
        {
            "command": [
                "sleep",
                "0"
            ],
            "slot": "task",
            "count": {
                "per_slot": 1
            }
        }
    ],
    "attributes": {
        "system": {
            "duration": 0,
            "environment": {
                "TMPDIR": "/tmp/"
            }
        }
    },
    "version": 1
}


### Playing with the Synchronous job submission API

One slight hiccup to creating jobspecs in Python is that some attributes of a jobspec are set in the _initializer_ (`.from_command` or another method) while others are specified by methods to the jobspec itself. Below is an example, and here's a brief table for reference:

| Operation                                      | CLI                                                                       | Python                                                                                     |
|------------------------------------------------|---------------------------------------------------------------------------|--------------------------------------------------------------------------------------------|
| Setting nodes, tasks, and cores                | -N _n_, -n _n_, and -c _n_, respectively                                  | in the initializer: num_nodes, num_tasks, num_cores                                        |
| Setting node exclusivity                       | `-x` or enforced by policy                                                | in the initializer: exclusive=True                                                         |
| Setting time limits                            | `-t MINUTES\|FSD`                                                         | `jobspec.duration=[seconds\|FSD]`                                                          |
| Set working directory                          | Automatically set to current directory or `--cwd=`                        | `jobspec.cwd=[string of path]`                                                             |
| Set environment                                | Default set to current environment, or `--env=[modifier]`                 | `jobspec.environment=[dict]`                                                               |
| Setting stdout and stderr                      | `--output [OUT] --error [ERR]` or your terminal by default for alloc/run  | `jobspec.stdout = <path>` or `jobspec.stderr = <path>`                                                     |
| Setting system attributes                      | `-S KEY[=VAL]`                                                            | `jobspec.setattr(key, val)`                                                                |
| Set shell attributes                           | `-o KEY[=VAL]`                                                            | `jobspec.setattr_shell_option(key, val)`                                                   |
| Modifying configuration of a subinstance       | `--conf` for batch/alloc                                                  | in the initializer, requires `.from_batch_command` or `.from_nest_command`                 |

The example below is taken from the [El Cap documentation maintained by Ramesh Pankajakshan](https://hpc.llnl.gov/documentation/user-guides/using-el-capitan-systems/introduction-and-quickstart/flux).

In [39]:
# Create a Flux handle (discovers the FLUX_URI in the environment)
handle = flux.Flux()
jobspec = JobspecV1.from_command(
    command=["sleep", "5"], num_tasks=4, num_nodes=1,
)
jobspec.cwd = os.getcwd()
jobspec.exclusive=1
jobspec.duration="5m"
jobspec.environment = {"TMPDIR": "/tmp/"}
jobspec.stdout="PYEXAMPLE.{{id}}.out"
jobspec.stderr="PYEXAMPLE.{{id}}.err"
id = flux.job.submit(handle, jobspec)

info = flux.job.event_wait(handle, id, "finish")
print(info)

1754855213.80323: finish {'status': 0}


### `FluxExecutor`

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> The `FluxExector` class makes it easy to do bulk submission in Python
</div>

We can use the FluxExecutor class to submit large numbers of jobs to Flux. This method resembles python's `concurrent.futures` interface.

In [40]:
from flux.job import FluxExecutor

with FluxExecutor() as executor:
    jobspec = JobspecV1.from_command(["sleep", "3"])
    jobspec.environment = {"TMPDIR": "/tmp/"}
    futures = [executor.submit(jobspec) for _ in range(10)]
    # wait for the jobid for each job, as a proxy for the job being submitted

# all jobs submitted - print timings
print("I'm all done")

I'm all done


In [41]:
# Submit the FluxExecutor based script.
!flux python bulksubmit_executor.py -n200 /bin/sleep 0

bulksubmit_executor: submitted 200 jobs in 0.18s. 1118.31job/s
bulksubmit_executor: First job finished in about 0.381s
|██████████████████████████████████████████████████████████| 100.0% (310.2 job/s)
bulksubmit_executor: Ran 200 jobs in 0.8s. 249.6 job/s


In [42]:
# Here is how to use concurrent futures
import concurrent.futures

jobspec = flux.job.JobspecV1.from_command(["/bin/true"])
jobspec.environment = {"TMPDIR": "/tmp/"}
with flux.job.FluxExecutor() as executor:
    futures = [executor.submit(jobspec) for _ in range(5)]
    for f in concurrent.futures.as_completed(futures):
            print(f.result())

0
0
0
0
0


### `flux.event_watch` 

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> The `flux.job.event_watch` function makes it easy to watch events for a job
</div>

If you want to get the output of a job (or more generally, stream events) you can do that as follows. Let's submit a quick job, and then look at the output.


In [27]:
# Create the Jobspec from a command to run a python script, and specify resources
f = flux.Flux()

joke = "I have two Dobermans, Rolex and Timex. They are watch dogs."
jobspec = flux.job.JobspecV1.from_command(command=["echo", joke], num_tasks=1, num_nodes=1, cores_per_task=1)
jobspec.environment = {"TMPDIR": "/tmp/"}
jobid = flux.job.submit(f, jobspec, waitable=True)

# Wait until the job finishes
flux.job.wait(f, jobid)
print(jobid)

# Wait on an event to complete and then print its associated data
for line in flux.job.event_watch(f, jobid, "guest.output"):
    print(line)

ƒabUbzYD5
1754853825.08733: header {'version': 1, 'encoding': {'stdout': 'UTF-8', 'stderr': 'UTF-8'}, 'count': {'stdout': 1, 'stderr': 1}, 'options': {}}
1754853825.09276: data {'stream': 'stderr', 'rank': '0', 'eof': True}
1754853825.09278: data {'stream': 'stdout', 'rank': '0', 'data': 'I have two Dobermans, Rolex and timex. They are watch dogs.\n'}
1754853825.09279: data {'stream': 'stdout', 'rank': '0', 'eof': True}


### `flux.job.JobOutputWatch`

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> Synchronously or asynchronously watch for job output with asynchronous job submission
</div>

#### Watch for job output with a Sychronous job submission

In [44]:
# Create the Jobspec from a command to run a python script, and specify resources
f = flux.Flux()

joke = "The inventor of the throat lozenges passed away. There was no coffin at his funeral."
jobspec = JobspecV1.from_command(command=["echo", joke], num_tasks=4, num_nodes=1)
jobspec.environment = {"TMPDIR": "/tmp/"}
jobid = flux.job.submit(f, jobspec)

t = flux.job.output.JobOutputWatchLines(f, jobid).getline()
print(t)

['stdout', 'The inventor of the throat lozenges passed away. There was no coffin at his funeral.']


#### Watch for job output with asynchronous job submission

In [45]:
import concurrent.futures
import flux.job

## Define something we want to happen when the futures are fulfilled
def print_output(fut):
    t = flux.job.output.JobOutputWatchLines(f, fut.jobid()).getline()
    for line in t:
        print(f'{fut.jobid()}: {line}')

## Submit all of the futures using the executor.
jobspec = flux.job.JobspecV1.from_command(["echo", "Wingardium Leviosa! ✨"])
jobspec.environment = {"TMPDIR": "/tmp/"}

with flux.job.FluxExecutor() as executor:
        futures = [executor.submit(jobspec) for _ in range(3)]
        for future in futures:
            future.add_done_callback(print_output)

ƒnrM71A1V: stdout
ƒnrM71A1V: Wingardium Leviosa! ✨
ƒnrM71A1W: stdout
ƒnrM71A1W: Wingardium Leviosa! ✨
ƒnrM71A1X: stdout
ƒnrM71A1X: Wingardium Leviosa! ✨


### `flux.job.job_list`

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> Get an entire listing of jobs via remote procedure call (rpc)
</div>


Finally, it can be really helpful to get an entire listing of jobs. You can do that as follows. Note that the `job_list` is creating a remote procedure call (rpc) and we call `get` to retrieve the output.

In [46]:
flux.job.job_list(f).get()

{'jobs': [{'id': 101237848539138,
   'userid': 1000,
   'urgency': 16,
   'priority': 16,
   't_submit': 1754855438.071756,
   't_depend': 1754855438.0827703,
   't_run': 1754855438.0959415,
   't_cleanup': 1754855438.1109397,
   't_inactive': 1754855438.1127431,
   'state': 64,
   'name': 'echo',
   'ntasks': 1,
   'ncores': 1,
   'duration': 0.0,
   'nnodes': 1,
   'ranks': '0',
   'nodelist': 'ip-10-0-20-251',
   'success': True,
   'exception_occurred': False,
   'result': 1,
   'expiration': 9223372036.0,
   'waitstatus': 0},
  {'id': 101237848539136,
   'userid': 1000,
   'urgency': 16,
   'priority': 16,
   't_submit': 1754855438.0714943,
   't_depend': 1754855438.0826428,
   't_run': 1754855438.0956306,
   't_cleanup': 1754855438.1106298,
   't_inactive': 1754855438.1125581,
   'state': 64,
   'name': 'echo',
   'ntasks': 1,
   'ncores': 1,
   'duration': 0.0,
   'nnodes': 1,
   'ranks': '0',
   'nodelist': 'ip-10-0-20-251',
   'success': True,
   'exception_occurred': False,
 

## Stream Events

<div class="alert alert-block" style="background-color:rebeccapurple; color: white">
<span style="font-weight:600">Description:</span> Use the `JournalConsumer` to stream events in real time
</div>

For this example, we recommend you open up two side-by-side terminals. In the first, create a `JournalConsumer`. Since we set the timeout to -1, we will get all events that have been seen by the instance and then it will wait for new events.

```bash
handle = flux.Flux()
consumer = flux.job.JournalConsumer(handle).start()
while True:
    event = consumer.poll(timeout=-1)
    print(event)
```
In the second terminal, try submitting a job. Anything will do!

```bash
flux run sleep 5
flux run echo "A slice of apple pie is 2:50 in Jamaica and 3:50 in the Bahamas. These are the pie rates of the Caribbean."
```
This is a powerful capability, because it allows for getting events for jobs as they happen, in real-time. We have built an entire state machine library to run complex workflows with this functionality.